# Calibration-policy ablation — "coverage is a calibration property, not a training one"

Re-calibrates the cached-feature head posteriors ONLY (no backbone retraining); reuses the conformal
harness across calibration ∈ {marginal_split, mondrian, shift_robust} × backbone × dataset ×
training_method × score × ρ_test × ≥3 seeds × ≥10 splits.

Hypotheses (report all): **C1** mondrian hits target for every method incl ERM while marginal_split is
significantly below (quantify shortfall) — group-conditional calibration, not the representation,
delivers worst-group coverage. **C2** under mondrian, worst-group set size differs by method and tracks
base accuracy (training buys efficiency, calibration buys coverage). **C3** Mondrian (cal ρ=0.95)
worst-group coverage stays valid (≥target−0.02) across the ρ sweep?

Deliverable: `CALIBRATION_ABLATION.md` + CSV (calibration column) + C1 figure. **STOP** for review.

## 0. Parameters — **EDIT THESE**

In [ ]:
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/octadion/vgscp.git"
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
SEEDS         = 3
N_SPLITS      = 10
CELEBA_RESNET_MAX_TRAIN = 30000
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE  = "kaggle"   # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE   = ""
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + repo + datasets (reuses cached features)

In [ ]:
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT; print("CelebA root:", CELEBA_ROOT)
else: print(f"[note] CelebA unavailable (source={CELEBA_SOURCE}) -> Waterbirds-only.")
for c in ("cache_clip", "cache_resnet", "study"):  # study Drive-backed so CSV persists for the CelebA append
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")

## 3. Build GridData per (backbone × dataset) — cache hit if the grid already ran

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224, "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

KEYS = [("waterbirds", "resnet50_erm"), ("waterbirds", "clip_vitb32")]
if CELEBA_OK: KEYS += [("celeba", "resnet50_erm"), ("celeba", "clip_vitb32")]
data, skipped = {}, []
for ds, bb in KEYS:
    try:
        t = time.time(); data[(bb, ds)] = build_griddata(ds, bb, cfg_for(ds), seed=0)
        print(f"[built] {bb}/{ds} ({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, ds)); print(f"[SKIP] {bb}/{ds}: {e}")
print("cells:", list(data.keys()), "| skipped:", skipped)

## 4. Run the ablation (calibration × method × score × ρ × seed × split) + C1/C2/C3 → STOP

In [ ]:
from study_robust_train.calibration_ablation import (run_ablation, write_csv,
    write_calibration_ablation_md, make_c1_figure)

t = time.time()
out = run_ablation(data, seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)   # all 3 calibrations, APS/RAPS/THR, full rho sweep
print(f"[ablation] {len(out['records'])} records, {len(out['excluded'])} excluded ({(time.time()-t)/60:.1f} min)")
os.makedirs("results/study", exist_ok=True)
write_csv(out["records"], "results/study/calibration_ablation.csv")
figs = make_c1_figure(out, "results/study/figures")
write_calibration_ablation_md(out, "CALIBRATION_ABLATION.md", fig_paths=figs)
print("wrote CALIBRATION_ABLATION.md, results/study/calibration_ablation.csv,", len(figs), "figures")

for key, v in out["verdicts"].items():
    c1, c2, c3 = v["C1"], v["C2"], v["C3"]
    print(f"\n{key}:")
    print(f"  C1 holds (mondrian hits target for all + marginal_split below for all): {c1['C1_holds']}")
    for m, row in c1["methods"].items():
        print(f"    {m}: mondrian={row['mondrian']['mean']:.3f}  marginal_split={row['marginal_split']['mean']:.3f}"
              f"  (shortfall {row['split_shortfall']:+.3f})")
    print(f"  C2 efficiency tracks accuracy: {c2['efficiency_tracks_accuracy']} (corr {c2['acc_vs_setsize_corr']:.3f})")
    surv = {m: {sc: c3['methods'][m][sc]['survives'] for sc in c3['methods'][m]} for m in c3['methods']}
    print(f"  C3 mondrian survives shift: {surv}")

## 5. Show CALIBRATION_ABLATION.md + C1 figure

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("CALIBRATION_ABLATION.md", encoding="utf-8").read()))
for p in figs: display(Image(p))

## 6. STOP — ablation complete
This earns the thesis "worst-group coverage is delivered by the calibration mechanism, not the
training/representation" if C1 holds. Hand C1/C2/C3 to the researcher. **STOP for human review** —
do NOT run the heavy full-GroupDRO fine-tune or any 3rd/4th dataset.